In [4]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, window, avg


Create a `SparkSession` running locally with 12 threads and enable Hive support.

In [5]:
spark = SparkSession.builder \
        .appName('odais_assignment') \
        .master('local[12]') \
        .enableHiveSupport() \
        .getOrCreate()

In [6]:
spark



Set `spark.sql.shuffle.partitions = 12` to control the number of shuffle partitions used by aggregations and joins.

In [7]:
spark.conf.set("spark.sql.shuffle.partitions",12)


Define a schema

In [8]:
s = "event_timestamp STRING,country STRING,temperature DOUBLE"


Create a streaming DataFrame `df`

In [9]:
df= spark.readStream.format('json')\
    .option('path','file:///home/itversity/itversity-material/latest-odai/lab2/batches/')\
    .schema(s)\
    .load()


Cast `event_timestamp` to a proper `Timestamp` as `event_time` and select `country` and `temperature` into `df_trans` for time-based processing.

In [10]:
df_trans=df.select(col('event_timestamp').cast("Timestamp").alias('event_time') ,'country','temperature')



Create `df_agg` by applying a 15-minute watermark on `event_time`, grouping by a 15-minute window and `country`, computing the average temperature, and selecting `window_start`, `window_end`, `country`, and `avg_temperature`.

In [11]:
df_agg=df_trans.withWatermark('event_time', '15 minutes')\
        .groupby(window(col("event_time"), "15 minutes"),col("country"))\
        .agg(avg('temperature').alias('avg_temperature'))\
        .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("country"),
        col("avg_temperature")
    )


Define `write_to_multiple_sinks(df_agg, batch_id)` to write each micro-batch both to Parquet files and to a Hive table `default.hive_temperature_avg`. This function runs per batch when using `foreachBatch`.

In [12]:
def write_to_multiple_sinks(df_agg, batch_id):
    df_agg.write \
        .format("parquet") \
        .mode("append") \
        .save("file:///home/itversity/itversity-material/latest-odai/lab2/temp/")
        
    df_agg.write \
        .format("hive") \
        .mode("append") \
        .saveAsTable("default.hive_temperature_avg")

In [13]:
query = df_agg.writeStream \
    .foreachBatch(write_to_multiple_sinks) \
    .outputMode("update") \
    .start()

In [15]:
query.stop()